# ASC Phase 2 — Roberta-base Colab

Bản này chạy trực tiếp với data high-confidence hiện tại:

```text
sentence_text: string
aspects: array<string>
sentiments: array<array<double>>
```

Trong đó:

```text
sentiments[i] = [prob_negative, prob_neutral, prob_positive]
```

Không đổi format data gốc. Khi flatten, code explode `aspects + sentiments`, rồi lấy `argmax(sentiments)` làm nhãn:

```text
0 = negative
1 = neutral
2 = positive
```

Flow:
1. Download/load 5 file high-confidence.
2. Flatten sentence-level thành pair-level `(sentence, aspect)`.
3. Sample 400k pair/category:
   - lấy toàn bộ neutral.
   - còn lại lấy pos/neg theo tỉ lệ 1:1.
4. Merge pseudo + gold train.
5. Remove leakage với gold test theo `sentence + aspect`.
6. Train `roberta-base`.
7. Benchmark trên gold test.


In [ ]:
!pip -q install pyspark==3.5.1 gdown pyarrow pandas datasets transformers accelerate evaluate scikit-learn sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
from pathlib import Path

RANDOM_SEED = 2026
N_PER_CATEGORY = 400_000

PROJECT_DIR = "/content/drive/MyDrive/asc_phase2"
RAW_DIR = f"{PROJECT_DIR}/raw_high_conf"
GOLD_DIR = f"{PROJECT_DIR}/gold"
OUTPUT_DIR = f"{PROJECT_DIR}/outputs"
MODEL_DIR = f"{PROJECT_DIR}/asc_teacher_phase1/asc_teacher_phase1"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(GOLD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

PAIR_LEVEL_ALL_PATH = f"{OUTPUT_DIR}/pair_level_all"
PSEUDO_SAMPLED_BY_CAT_PATH = f"{OUTPUT_DIR}/pseudo_sampled_by_category"
PSEUDO_SAMPLED_PATH = f"{OUTPUT_DIR}/pseudo_sampled_2m"
FINAL_TRAIN_PATH = f"{OUTPUT_DIR}/final_train_no_leakage"
REPORT_PATH = f"{OUTPUT_DIR}/sampling_report.json"
TEST_METRICS_PATH = f"{OUTPUT_DIR}/gold_test_metrics.json"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)


Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/asc_phase2
OUTPUT_DIR: /content/drive/MyDrive/asc_phase2/outputs
MODEL_DIR: /content/drive/MyDrive/asc_phase2/asc_teacher_phase1/asc_teacher_phase1


In [ ]:

HIGH_CONF_GDRIVE_IDS = {
    "electronics_p1" : "1etn4E1vW7Ab_jKVi1nTJpSjooEamCPbP",
    "electronics_p2" : "1Z-cRLF21uaUBgoRpE5nYPVns-XHqgX0k",
    "software"       : "1SIahLZ81-fvqtwSYeGOLuzbxIiySw36d",
    "kindle_store"   : "1eeKvI9H7nUaVaIKdDM1yGTGbDy2sHdB1",
    "office_products": "1yEdq7SGfP-62h4dvKXOF63Q4DzP1LuBf",
}


ASC_HIGH_PATHS = {
    name: f"{RAW_DIR}/{name}.parquet"
    for name in HIGH_CONF_GDRIVE_IDS.keys()
}

GOLD_TRAIN_PATH = f"{GOLD_DIR}/gold_train.csv"
GOLD_TEST_PATH = f"{GOLD_DIR}/gold_test.csv"

ASC_HIGH_PATHS


{'electronics_p1': '/content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet',
 'electronics_p2': '/content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p2.parquet',
 'software': '/content/drive/MyDrive/asc_phase2/raw_high_conf/software.parquet',
 'kindle_store': '/content/drive/MyDrive/asc_phase2/raw_high_conf/kindle_store.parquet',
 'office_products': '/content/drive/MyDrive/asc_phase2/raw_high_conf/office_products.parquet'}

In [ ]:
import os
import gdown

def download_gdrive_file(file_id: str, output_path: str, overwrite: bool = False):
    if os.path.exists(output_path) and not overwrite:
        print(f"[EXISTS] {output_path}")
        return output_path

    url = f"https://drive.google.com/uc?id={file_id}"
    print(f"[DOWNLOAD] {file_id} -> {output_path}")
    gdown.download(url, output_path, quiet=False, fuzzy=True)
    return output_path

for name, file_id in HIGH_CONF_GDRIVE_IDS.items():
    download_gdrive_file(file_id, ASC_HIGH_PATHS[name], overwrite=False)

print("Done.")


[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p2.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/software.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/kindle_store.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/office_products.parquet
Done.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce

spark = (
    SparkSession.builder
    .appName("ASC Phase 2 Roberta")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)


Spark: 3.5.1


In [ ]:
for category, path in ASC_HIGH_PATHS.items():
    print("=" * 100)
    print(category, path)
    df = spark.read.parquet(path)
    print("RAW COUNT:", df.count())
    df.printSchema()
    df.show(3, truncate=False)


electronics_p1 /content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet
RAW COUNT: 12990920
root
 |-- parent_asin: string (nullable = true)
 |-- sentence_id: integer (nullable = true)
 |-- sentence_text: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- category_name: string (nullable = true)
 |-- gate_confidence: double (nullable = true)
 |-- aspects: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- sentiments: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: double (containsNull = true)

+-----------+-----------+---------------------------------------------------------------------------------------+------+-----------------+------------------+---------------+--------------------------------------------------------------------+
|parent_asin|sentence_id|sentence_text                                                                          |rating|category_name    |gate_confidence   

In [ ]:
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

def first_existing_col(df: DataFrame, candidates: list[str]) -> str | None:
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

def normalize_label_expr(col):
    raw = F.lower(F.trim(col.cast("string")))
    return (
        F.when(raw.isin("negative", "neg", "-1", "0", "label_0"), F.lit("negative"))
         .when(raw.isin("neutral", "neu", "1", "label_1"), F.lit("neutral"))
         .when(raw.isin("positive", "pos", "2", "label_2"), F.lit("positive"))
         .when(raw.contains("negative"), F.lit("negative"))
         .when(raw.contains("neutral"), F.lit("neutral"))
         .when(raw.contains("positive"), F.lit("positive"))
         .when(raw.contains("neg"), F.lit("negative"))
         .when(raw.contains("neu"), F.lit("neutral"))
         .when(raw.contains("pos"), F.lit("positive"))
         .otherwise(raw)
    )

def add_label_id_from_text(df: DataFrame, label_col: str = "label") -> DataFrame:
    df = df.withColumn("label", normalize_label_expr(F.col(label_col)))
    return (
        df
        .filter(F.col("label").isin("negative", "neutral", "positive"))
        .withColumn(
            "label_id",
            F.when(F.col("label") == "negative", F.lit(0))
             .when(F.col("label") == "neutral", F.lit(1))
             .when(F.col("label") == "positive", F.lit(2))
             .otherwise(F.lit(None).cast("int"))
        )
    )

def make_sample_key(sentence_col="sentence", aspect_col="aspect"):
    return F.sha2(
        F.concat_ws(
            "||",
            F.lower(F.trim(F.col(sentence_col))),
            F.lower(F.trim(F.col(aspect_col))),
        ),
        256,
    )

def flatten_sentence_to_pairs(df: DataFrame, category: str) -> DataFrame:
    sentence_col = first_existing_col(
        df,
        ["sentence_text", "sentence", "text", "review_sentence", "review_text", "content"],
    )
    aspect_col = first_existing_col(
        df,
        ["aspects", "aspect_terms", "aspect_list", "targets", "terms"],
    )
    sentiment_col = first_existing_col(
        df,
        ["sentiments", "sentiment_probs", "probabilities", "probs", "scores"],
    )

    if sentence_col is None:
        raise ValueError(f"Không tìm thấy sentence column. Columns: {df.columns}")
    if aspect_col is None:
        raise ValueError(f"Không tìm thấy aspect array column. Columns: {df.columns}")
    if sentiment_col is None:
        raise ValueError(f"Không tìm thấy sentiment probability column. Columns: {df.columns}")

    gate_expr = F.col("gate_confidence").cast("double") if "gate_confidence" in df.columns else F.lit(None).cast("double")
    rating_expr = F.col("rating").cast("double") if "rating" in df.columns else F.lit(None).cast("double")
    parent_expr = F.col("parent_asin").cast("string") if "parent_asin" in df.columns else F.lit(None).cast("string")
    sid_expr = F.col("sentence_id").cast("string") if "sentence_id" in df.columns else F.lit(None).cast("string")

    pair_df = (
        df
        .withColumn("source_category", F.lit(category))
        .withColumn("_zipped", F.arrays_zip(F.col(aspect_col), F.col(sentiment_col)))
        .withColumn("_pair", F.explode_outer(F.col("_zipped")))
        .select(
            F.col("source_category").alias("category_name"),
            parent_expr.alias("parent_asin"),
            sid_expr.alias("sentence_id"),
            F.col(sentence_col).cast("string").alias("sentence"),
            F.col(f"_pair.{aspect_col}").cast("string").alias("aspect"),
            F.col(f"_pair.{sentiment_col}").alias("sentiment_probs"),
            rating_expr.alias("rating"),
            gate_expr.alias("gate_confidence"),
        )
        .filter(F.col("sentence").isNotNull())
        .filter(F.col("aspect").isNotNull())
        .filter(F.col("sentiment_probs").isNotNull())
        .filter(F.length(F.trim(F.col("sentence"))) > 0)
        .filter(F.length(F.trim(F.col("aspect"))) > 0)
        .withColumn("sentence", F.trim(F.col("sentence")))
        .withColumn("aspect", F.trim(F.col("aspect")))
    )

    pair_df = (
        pair_df
        .withColumn("prob_neg", F.col("sentiment_probs")[0].cast("double"))
        .withColumn("prob_neu", F.col("sentiment_probs")[1].cast("double"))
        .withColumn("prob_pos", F.col("sentiment_probs")[2].cast("double"))
        .filter(F.col("prob_neg").isNotNull())
        .filter(F.col("prob_neu").isNotNull())
        .filter(F.col("prob_pos").isNotNull())
        .withColumn(
            "label_id",
            F.when(
                (F.col("prob_neg") >= F.col("prob_neu")) &
                (F.col("prob_neg") >= F.col("prob_pos")),
                F.lit(0)
            )
            .when(
                (F.col("prob_neu") >= F.col("prob_neg")) &
                (F.col("prob_neu") >= F.col("prob_pos")),
                F.lit(1)
            )
            .otherwise(F.lit(2))
        )
        .withColumn(
            "label",
            F.when(F.col("label_id") == 0, F.lit("negative"))
             .when(F.col("label_id") == 1, F.lit("neutral"))
             .when(F.col("label_id") == 2, F.lit("positive"))
        )
        .withColumn("confidence", F.greatest("prob_neg", "prob_neu", "prob_pos"))
        .withColumn("sample_key", make_sample_key("sentence", "aspect"))
        .dropDuplicates(["sample_key", "label_id"])
    )

    return pair_df.select(
        "sample_key", "category_name", "parent_asin", "sentence_id",
        "sentence", "aspect", "label", "label_id",
        "confidence", "prob_neg", "prob_neu", "prob_pos",
        "rating", "gate_confidence",
    )


In [ ]:
# Test flatten trước
category = "electronics_p1"
path = ASC_HIGH_PATHS[category]

raw_df = spark.read.parquet(path)
df_pair = flatten_sentence_to_pairs(raw_df, category)

print("PAIR COUNT:", df_pair.count())
df_pair.show(10, truncate=False)
df_pair.groupBy("label_id", "label").count().orderBy("label_id").show(100, truncate=False)


PAIR COUNT: 15416279
+----------------------------------------------------------------+--------------+-----------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+--------+--------+------------------+---------------------+---------------------+--------------------+------+------------------+
|sample_key                                                      |category_name |parent_asin|sentence_id|sentence                                                                                                                                                                                     |aspect           |label   |label_id|confidence        |prob_neg             |prob_neu             |prob_pos            |rating|gate_confidence   |
+----------------------------------------------------------------+--------------+-----------+--

In [ ]:
def sample_exact_n_by_hash(df: DataFrame, n: int, seed: int, key_col: str = "sample_key") -> DataFrame:
    if n <= 0:
        return df.limit(0)

    total = df.count()
    if total <= n:
        return df

    fraction = min(1.0, (n * 1.5) / total)
    candidate = df.sample(False, fraction, seed=seed)

    return (
        candidate
        .withColumn("_rand_key", F.xxhash64(F.col(key_col), F.lit(seed)))
        .orderBy("_rand_key")
        .drop("_rand_key")
        .limit(n)
    )

def sample_phase2_category(df_pair: DataFrame, n_total: int = 400_000, seed: int = 42) -> tuple[DataFrame, dict]:
    neutral_df = df_pair.filter(F.col("label_id") == 1)
    pos_df = df_pair.filter(F.col("label_id") == 2)
    neg_df = df_pair.filter(F.col("label_id") == 0)

    n_neu_total = neutral_df.count()
    n_pos_total = pos_df.count()
    n_neg_total = neg_df.count()

    if n_neu_total >= n_total:
        print(f"[WARN] Neutral >= {n_total:,}; lấy {n_total:,} neutral, không lấy pos/neg.")
        sampled_neu = sample_exact_n_by_hash(neutral_df, n_total, seed)
        sampled_pos = pos_df.limit(0)
        sampled_neg = neg_df.limit(0)
        target_neu, target_pos, target_neg = n_total, 0, 0
    else:
        sampled_neu = neutral_df
        remaining = n_total - n_neu_total

        target_pos = remaining // 2
        target_neg = remaining - target_pos

        actual_target_pos = min(target_pos, n_pos_total)
        actual_target_neg = min(target_neg, n_neg_total)

        leftover = remaining - actual_target_pos - actual_target_neg

        if leftover > 0:
            add_pos = min(leftover, max(0, n_pos_total - actual_target_pos))
            actual_target_pos += add_pos
            leftover -= add_pos

        if leftover > 0:
            add_neg = min(leftover, max(0, n_neg_total - actual_target_neg))
            actual_target_neg += add_neg
            leftover -= add_neg

        target_neu = n_neu_total
        target_pos = actual_target_pos
        target_neg = actual_target_neg

        sampled_pos = sample_exact_n_by_hash(pos_df, int(target_pos), seed + 11)
        sampled_neg = sample_exact_n_by_hash(neg_df, int(target_neg), seed + 29)

    sampled = (
        sampled_neu
        .unionByName(sampled_pos, allowMissingColumns=True)
        .unionByName(sampled_neg, allowMissingColumns=True)
    )

    sampled = sample_exact_n_by_hash(sampled, n_total, seed + 101)

    final_counts = {
        ID2LABEL[int(r["label_id"])]: int(r["count"])
        for r in sampled.groupBy("label_id").count().collect()
    }

    stats = {
        "total_pairs_after_flatten": int(df_pair.count()),
        "available": {
            "negative": int(n_neg_total),
            "neutral": int(n_neu_total),
            "positive": int(n_pos_total),
        },
        "target": {
            "negative": int(target_neg),
            "neutral": int(target_neu),
            "positive": int(target_pos),
        },
        "sampled_pairs": int(sampled.count()),
        "final_label_counts": final_counts,
    }

    return sampled, stats


In [ ]:
SAVE_PAIR_LEVEL_ALL = False

# Nếu chạy lại từ đầu thì mở comment:
# !rm -rf "{PAIR_LEVEL_ALL_PATH}" "{PSEUDO_SAMPLED_BY_CAT_PATH}" "{PSEUDO_SAMPLED_PATH}" "{FINAL_TRAIN_PATH}"

sampled_paths = []
sampling_report = []

for category, path in ASC_HIGH_PATHS.items():
    print("\n" + "=" * 100)
    print(f"[PROCESS] {category}")
    print(f"[PATH] {path}")

    raw_df = spark.read.parquet(path)
    df_pair = flatten_sentence_to_pairs(raw_df, category)

    total_pairs = df_pair.count()
    print(f"[PAIR LEVEL] {category}: {total_pairs:,}")

    if SAVE_PAIR_LEVEL_ALL:
        (
            df_pair
            .repartition(8)
            .write
            .mode("append")
            .parquet(PAIR_LEVEL_ALL_PATH)
        )

    sampled_cat, stats = sample_phase2_category(
        df_pair,
        n_total=N_PER_CATEGORY,
        seed=RANDOM_SEED,
    )

    sampled_count = sampled_cat.count()
    print(f"[SAMPLED] {category}: {sampled_count:,}")

    out_cat_path = f"{PSEUDO_SAMPLED_BY_CAT_PATH}/{category}"

    (
        sampled_cat
        .repartition(4)
        .write
        .mode("overwrite")
        .parquet(out_cat_path)
    )

    sampled_paths.append(out_cat_path)

    stats.update({
        "category_name": category,
        "input_path": path,
        "seed": RANDOM_SEED,
    })
    sampling_report.append(stats)

    print(json.dumps(stats, ensure_ascii=False, indent=2))

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(sampling_report, f, ensure_ascii=False, indent=2)

print("Report:", REPORT_PATH)



[PROCESS] electronics_p1
[PATH] /content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet
[PAIR LEVEL] electronics_p1: 15,416,279


In [ ]:
pseudo_df = spark.read.parquet(*sampled_paths)
pseudo_df = pseudo_df.dropDuplicates(["sample_key", "label_id"])

(
    pseudo_df
    .repartition(8)
    .write
    .mode("overwrite")
    .parquet(PSEUDO_SAMPLED_PATH)
)

print("Saved pseudo:", PSEUDO_SAMPLED_PATH)
print("Pseudo count:", pseudo_df.count())
pseudo_df.groupBy("category_name").count().orderBy("category_name").show(100, truncate=False)
pseudo_df.groupBy("label_id", "label").count().orderBy("label_id").show(100, truncate=False)


In [ ]:
def read_table_spark(path: str) -> DataFrame:
    if path.endswith(".parquet"):
        return spark.read.parquet(path)
    if path.endswith(".csv"):
        return spark.read.option("header", True).csv(path)
    raise ValueError(f"Chỉ hỗ trợ parquet/csv: {path}")

def standardize_gold_df(df: DataFrame, category_name: str) -> DataFrame:
    sentence_col = first_existing_col(df, ["sentence", "sentence_text", "text", "review_sentence", "review_text", "content"])
    aspect_col = first_existing_col(df, ["aspect", "aspect_term", "target", "term"])
    label_col = first_existing_col(df, ["label", "sentiment", "polarity", "sentiment_label", "label_id"])

    if sentence_col is None or aspect_col is None or label_col is None:
        raise ValueError(f"Gold data cần sentence/aspect/label. Columns: {df.columns}")

    out = (
        df
        .select(
            F.lit(category_name).alias("category_name"),
            F.col(sentence_col).cast("string").alias("sentence"),
            F.col(aspect_col).cast("string").alias("aspect"),
            F.col(label_col).cast("string").alias("label"),
        )
        .filter(F.col("sentence").isNotNull())
        .filter(F.col("aspect").isNotNull())
        .filter(F.length(F.trim(F.col("sentence"))) > 0)
        .filter(F.length(F.trim(F.col("aspect"))) > 0)
        .withColumn("sentence", F.trim(F.col("sentence")))
        .withColumn("aspect", F.trim(F.col("aspect")))
    )

    out = add_label_id_from_text(out, "label")

    return (
        out
        .withColumn("sample_key", make_sample_key("sentence", "aspect"))
        .select("sample_key", "category_name", "sentence", "aspect", "label", "label_id")
        .dropDuplicates(["sample_key", "label_id"])
    )

gold_train_raw = read_table_spark(GOLD_TRAIN_PATH)
gold_test_raw = read_table_spark(GOLD_TEST_PATH)

gold_train_df = standardize_gold_df(gold_train_raw, "gold_train")
gold_test_df = standardize_gold_df(gold_test_raw, "gold_test")

print("Gold train:", gold_train_df.count())
print("Gold test:", gold_test_df.count())
gold_train_df.groupBy("label_id", "label").count().orderBy("label_id").show()
gold_test_df.groupBy("label_id", "label").count().orderBy("label_id").show()


In [ ]:
gold_test_keys = gold_test_df.select("sample_key").dropDuplicates()

pseudo_no_leak = pseudo_df.join(gold_test_keys, on="sample_key", how="left_anti")
gold_train_no_leak = gold_train_df.join(gold_test_keys, on="sample_key", how="left_anti")

final_train_df = (
    pseudo_no_leak
    .select("sample_key", "category_name", "sentence", "aspect", "label", "label_id")
    .unionByName(
        gold_train_no_leak.select("sample_key", "category_name", "sentence", "aspect", "label", "label_id"),
        allowMissingColumns=True,
    )
    .dropDuplicates(["sample_key", "label_id"])
)

(
    final_train_df
    .repartition(8)
    .write
    .mode("overwrite")
    .parquet(FINAL_TRAIN_PATH)
)

print("Pseudo before leakage removal:", pseudo_df.count())
print("Pseudo no leak:", pseudo_no_leak.count())
print("Gold train before leakage removal:", gold_train_df.count())
print("Gold train no leak:", gold_train_no_leak.count())
print("Final train:", final_train_df.count())
final_train_df.groupBy("label_id", "label").count().orderBy("label_id").show()


# Train roberta-base

In [ ]:
from datasets import load_dataset, DatasetDict

GOLD_TEST_STD_PATH = f"{OUTPUT_DIR}/gold_test_standardized"

(
    gold_test_df
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(GOLD_TEST_STD_PATH)
)

hf_data = load_dataset(
    "parquet",
    data_files={
        "train": f"{FINAL_TRAIN_PATH}/*.parquet",
        "test": f"{GOLD_TEST_STD_PATH}/*.parquet",
    },
)

hf_data


In [ ]:
def to_model_example(ex):
    return {
        "sentence": str(ex["sentence"]),
        "aspect": str(ex["aspect"]),
        "labels": int(ex["label_id"]),
    }

train_hf = hf_data["train"].map(
    to_model_example,
    remove_columns=hf_data["train"].column_names,
)

test_hf = hf_data["test"].map(
    to_model_example,
    remove_columns=hf_data["test"].column_names,
)

dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf,
})

split = dataset["train"].train_test_split(
    test_size=0.02,
    seed=RANDOM_SEED,
    stratify_by_column="labels",
)

dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": dataset["test"],
})

dataset


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"
MAX_LENGTH = 192

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_fn(batch):
    return tokenizer(
        batch["sentence"],
        batch["aspect"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["sentence", "aspect"],
)

tokenized


In [ ]:
import numpy as np
import evaluate

from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

accuracy = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision_macro": precision_metric.compute(predictions=preds, references=labels, average="macro", zero_division=0)["precision"],
        "recall_macro": recall_metric.compute(predictions=preds, references=labels, average="macro", zero_division=0)["recall"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2},
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
test_metrics = trainer.evaluate(tokenized["test"])
print(test_metrics)

with open(TEST_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print("Saved metrics:", TEST_METRICS_PATH)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

pred_out = trainer.predict(tokenized["test"])
preds = np.argmax(pred_out.predictions, axis=-1)
y_true = pred_out.label_ids

target_names = ["negative", "neutral", "positive"]

print(classification_report(y_true, preds, target_names=target_names, digits=4))

cm = confusion_matrix(y_true, preds, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
cm_df


In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

with open(f"{MODEL_DIR}/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": {"negative": 0, "neutral": 1, "positive": 2},
            "id2label": {"0": "negative", "1": "neutral", "2": "positive"},
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "seed": RANDOM_SEED,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved model:", MODEL_DIR)


In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0,
)

examples = [
    ("The screen is beautiful but the battery is terrible.", "screen"),
    ("The screen is beautiful but the battery is terrible.", "battery"),
]

for sent, asp in examples:
    print("=" * 80)
    print("sentence:", sent)
    print("aspect:", asp)
    print(clf({"text": sent, "text_pair": asp}))


In [ ]:
# spark.stop()
